# Dual Language Translator: Part 2 English -> Hindi

###  `Problem statement: Dual Language Translator: Description: Make a machine learning model with a feature that translates English words into both French and Hindi simultaneously. This feature should only translate English words or lines that have 10 or more letters. If an English word has fewer than 10 letters, the model should prompt the user to “upload again.” Guidelines: You have to train your own machine learning model. You should have a GUI for this task. The GUI should include an input section for entering English words and an output section for displaying the translated French and Hindi words.`

## Import all the libraries

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import pandas as pd

## Load all the dataset

In [3]:
data = pd.read_csv("english_hindi_dataset.csv")
data = data.dropna()

eng_texts = data["English"].values
hindi_texts = data["Hindi"].values

# Add <start> and <end> tokens for Hindi sentences
hindi_texts = ["<start> " + text + " <end>" for text in hindi_texts]
data

,English,Hindi
0,Where do you live,आप कहाँ रहते हैं
1,Thank you,धन्यवाद
2,Thank you very much,बहुत बहुत धन्यवाद
3,Stand up,खड़े हो जाओ
4,I am fine,मैं ठीक हूँ
...,...,...
1015,I am fine,मैं ठीक हूँ
1016,Please help me,कृपया मेरी मदद करें
1017,Good luck,शुभकामनाएँ
1018,Open the door,दरवाज़ा खोलो


## Preprocessing the data

In [5]:
# Tokenization & Padding

num_samples = len(eng_texts)
max_encoder_seq_length = 20
max_decoder_seq_length = 20

# Tokenize English
eng_tokenizer = Tokenizer(filters='', lower=True)
eng_tokenizer.fit_on_texts(eng_texts)
eng_sequences = eng_tokenizer.texts_to_sequences(eng_texts)
eng_input_data = pad_sequences(eng_sequences, maxlen=max_encoder_seq_length, padding='post')

# Tokenize Hindi
hin_tokenizer = Tokenizer(filters='', lower=True)
hin_tokenizer.fit_on_texts(hindi_texts)
hin_sequences = hin_tokenizer.texts_to_sequences(hindi_texts)
hin_input_data = pad_sequences(hin_sequences, maxlen=max_decoder_seq_length, padding='post')

# Decoder target data is shifted by one 
hin_target_data = np.zeros_like(hin_input_data)
hin_target_data[:, :-1] = hin_input_data[:, 1:]

num_encoder_tokens = len(eng_tokenizer.word_index) + 1
num_decoder_tokens = len(hin_tokenizer.word_index) + 1


## Buidling the model

In [6]:
latent_dim = 256  

# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(num_encoder_tokens, 128)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(num_decoder_tokens, 128)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Full Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, None)]       0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, None)]       0           []                               
                                                                                                  
 embedding (Embedding)          (None, None, 128)    11136       ['input_1[0][0]']                
                                                                                                  
 embedding_1 (Embedding)        (None, None, 128)    12544       ['input_2[0][0]']                
                                                                                              

In [7]:
# Train the Model

history = model.fit(
    [eng_input_data, hin_input_data],
    np.expand_dims(hin_target_data, -1),
    batch_size=32,
    epochs=30,
    validation_split=0.2
)


Epoch 1/30
26/26 [==============================] - 12s 87ms/step - loss: 1.3063 - accuracy: 0.7676 - val_loss: 0.8275 - val_accuracy: 0.8223
Epoch 2/30
26/26 [==============================] - 1s 28ms/step - loss: 0.7722 - accuracy: 0.8216 - val_loss: 0.7262 - val_accuracy: 0.8243
Epoch 3/30
26/26 [==============================] - 1s 25ms/step - loss: 0.6698 - accuracy: 0.8426 - val_loss: 0.6257 - val_accuracy: 0.8512
Epoch 4/30
26/26 [==============================] - 1s 26ms/step - loss: 0.5719 - accuracy: 0.8632 - val_loss: 0.5687 - val_accuracy: 0.8745
Epoch 5/30
26/26 [==============================] - 1s 32ms/step - loss: 0.4833 - accuracy: 0.8778 - val_loss: 0.4439 - val_accuracy: 0.8806
Epoch 6/30
26/26 [==============================] - 1s 31ms/step - loss: 0.3992 - accuracy: 0.8901 - val_loss: 0.3812 - val_accuracy: 0.8980
Epoch 7/30
26/26 [==============================] - 1s 24ms/step - loss: 0.3355 - accuracy: 0.9053 - val_loss: 0.3113 - val_accuracy: 0.9137
Epoch 8/30
2

# Inference model for predictions

In [8]:
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs)
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_states2 = [state_h2, state_c2]

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

In [9]:
reverse_target_word_index = hin_tokenizer.index_word
target_word_index = hin_tokenizer.word_index

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_word_index['<start>']

    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, '')

        if sampled_word == '<end>' or len(decoded_sentence.split()) > max_decoder_seq_length:
            stop_condition = True
        else:
            decoded_sentence += ' ' + sampled_word

        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return decoded_sentence.strip()

In [10]:
eng_input = ("Where do you live").strip()
seq = eng_tokenizer.texts_to_sequences([eng_input])
seq = pad_sequences(seq, maxlen=max_encoder_seq_length, padding='post')
translated = decode_sequence(seq)
print("Predicted Hindi:", translated)

1/1 [==============================] - 0s 135ms/step
Predicted Hindi: आप कहाँ रहते हैं


## Saving the model and tokenizers

In [22]:
import json

# Save model
model.save("eng_hindi_translation_model.keras")

# Save tokenizers
with open("english2_tokenizer.json", "w", encoding="utf-8") as f:
    f.write(eng_tokenizer.to_json())

with open("hindi_tokenizer.json", "w", encoding="utf-8") as f:
    f.write(hin_tokenizer.to_json())



encoder_model.save("encoder_model.keras")
decoder_model.save("decoder_model.keras")

# Save sequence lengths
import pickle
with open("seq_lengths.pkl", "wb") as f:
    pickle.dump((max_encoder_seq_length, max_decoder_seq_length), f)